# Лабораторная работа №13. HTTP: приложение наконец говорит

До этого мы изучали доставку. Теперь посмотрим application protocol: request line, headers, status code и body — и увидим, как всё это вложено в TCP/IP/Ethernet.

## Схема

![HTTP](assets/http.svg)

`web1` уже запускает простой HTTP server на `10.13.0.20:8080`. `pc1` — client.

## 1. Настроим client и проверим L3

На `pc1`:

```bash
ip addr flush dev eth1
ip addr add 10.13.0.10/24 dev eth1
ping -c 1 10.13.0.20
```

Если ping не проходит, `curl` failure ещё ничего не говорит об HTTP.

### Q1. Какие уровни должны уже работать до того, как HTTP request вообще сможет попасть в server process?

`введите ваш ответ`

## 2. curl показывает диалог

```bash
curl -v http://10.13.0.20:8080/
```

Отделите:

```text
TCP connection
HTTP request line + request headers
HTTP status line + response headers
body
```

### Q2. Чем status code `200` отличается от факта, что TCP connection успешно установился?

`введите ваш ответ`

## 3. Отправим HTTP руками

```bash
printf 'GET / HTTP/1.1\r\nHost: lab.example\r\nConnection: close\r\n\r\n' | nc 10.13.0.20 8080
```

Здесь видно, что HTTP/1.1 — текстовый protocol поверх byte stream TCP.

In [ ]:
%%capture_traffic pc1:eth1 --filter "tcp.port == 8080" --timeout 12 --packets 200 --save captures/http.pcap
import time
print("На pc1 выполните curl -v http://10.13.0.20:8080/")
time.sleep(10)

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
viewer = PcapViewer("captures/http.pcap", "tcp.port == 8080", limit=80)
frames = viewer.packets()
display(frames)
http = frames[frames["Protocol"].astype(str).str.contains("HTTP", case=False, na=False)] if not frames.empty else frames
if not http.empty:
    print(viewer.packet(int(http.iloc[0]["№"]), "protocols"))

### Q3. Какие части запроса HTTP можно увидеть в capture без TLS и какие из них перестали бы быть читаемыми после шифрования HTTPS?

`введите ваш ответ`

## 4. Что всё ещё видно при HTTPS

TLS скрывает HTTP method, path, headers и body от пассивного наблюдателя. Но остаются видимыми нижележащие metadata: IP endpoints, TCP ports, размеры/timing packets и сам факт TLS session. Некоторые TLS metadata зависят от версии и настроек.

## CHECK

Checker выполнит реальный HTTP GET к `web1:8080` и проверит ожидаемый marker в body.

## Итог

HTTP не заменяет TCP/IP/Ethernet — он использует их. Application status и transport connectivity отвечают на разные вопросы, поэтому troubleshooting идёт снизу вверх.